In [4]:
%pip uninstall -y tensorflow tensorflow-intel keras
%pip install --no-cache-dir tensorflow

Found existing installation: tensorflow 2.21.0
Uninstalling tensorflow-2.21.0:
  Successfully uninstalled tensorflow-2.21.0
Found existing installation: keras 3.12.3
Uninstalling keras-3.12.3:
  Successfully uninstalled keras-3.12.3
Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/350.6 MB ? eta -:--:--
   ---------------------------------------- 1.8/350.6 MB 10.1 MB/s eta 0:00:35
    --------------------------------------- 4.7/350.6 MB 11.9 MB/s eta 0:00:30
    --------------------------------------- 7.9/350.6 MB 13.2 MB/s eta 0:00:27
   - -------------------------------------- 11.0/350.6 MB 13.8 MB/s eta 0:00:25
   - -------------------------------------- 14.2/350.6 MB 14.3 MB/s eta 0:00:24
   -- ------------------------------------- 17.6/350.6 MB 14.4 MB/s eta 0:00:24
   -- ------------------------------------- 20.7/350.6 MB 14.5 MB/s eta 0:00:23
   -- ------------------------------------- 23.6/350.6 MB 14.5 MB/s eta 0:00:23
   --- ------------------------------------ 26.5/350.6 MB 14.5 MB/s eta 0:00:23
   --- ------------------------------------ 29.4/350.6 MB 14.3 MB/s eta 0:00:23
   --- ------------------------------------ 32.2/350.6 MB 14.3 MB/s eta 0:00:23
   ---- ----------------------------------- 35.7/350

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import tensorflow.keras as keras
import random

In [2]:
def build_q_network(state_shape, action_size):
    model = tf.keras.Sequential([
        layers.Input(shape=(state_shape, )),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(action_size, activation=None) #Outputs Q-values for each action
    ])
    return model

In [3]:
class ReplayBuffer:
    def __init__(self):
        self.buffer = []
    def store(self, state, action, reward, next_state, legal_actions, done):
        if len(self.buffer) == 5000:
            self.buffer.pop(0)
        self.buffer.append((state, action, reward, next_state, legal_actions, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, many_legal_actions, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), list(many_legal_actions), np.array(dones, dtype=np.float32))

In [39]:
def traded_resources(identifier):
  collection = ['w', 'h', 'o', 't', 'b']
  if identifier < 4:
    r1 = 0
  elif identifier < 8:
    r1 = 1
    identifier = identifier - 4
  elif identifier < 12:
    r1 = 2
    identifier = identifier - 8
  elif identifier < 16:
    r1 = 3
    identifier = identifier - 12
  else:
    r1 = 4
    identifier = identifier - 16

  new_collection = collection.copy()
  new_collection.remove(collection[r1])
  res2 = new_collection[identifier]
  r2 = collection.index(res2)
  return [r1, r2] #r1 is the resource being demanded, r2 is the resource being given


def conduct_trade(a1, a2, proposal, flag):
    if not flag:
        return False
    if a1.resources[proposal[2]] == 0:
      return False
    a1.resources[proposal[1]] = a1.resources[proposal[1]] + 1
    a2.resources[proposal[2]] = a2.resources[proposal[2]] + 1
    a1.resources[proposal[2]] = a1.resources[proposal[2]] - 1
    a2.resources[proposal[1]] = a2.resources[proposal[1]] - 1
    a1.loyalty[a2.id] = np.clip(a1.loyalty[a2.id] + 5, None, 100)
    a2.loyalty[a1.id] = np.clip(a2.loyalty[a1.id] + 3, None, 100)
    a1.last_traded = a2.id
    a2.last_traded = a1.id
    return True
    
def get_legal_actions(res):
    legal_actions = [0]
    if res[0] > 0:
      legal_actions.extend([5, 9, 13, 17]) #1, 5, 9, 13, 17
      legal_actions.extend([25, 29, 33, 37])
      legal_actions.extend([45, 49, 53, 57])
      legal_actions.extend([65, 69, 73, 77])
    if res[1] > 0:
      legal_actions.extend([1, 10, 14, 18])
      legal_actions.extend([21, 30, 34, 38])
      legal_actions.extend([41, 50, 54, 58])
      legal_actions.extend([61, 70, 74, 78])
    if res[2] > 0:
      legal_actions.extend([2, 6, 15, 19])
      legal_actions.extend([22, 26, 35, 39])
      legal_actions.extend([42, 46, 55, 59])
      legal_actions.extend([62, 66, 75, 79])
    if res[3] > 0:
      legal_actions.extend([3, 7, 11, 20])
      legal_actions.extend([23, 27, 31, 40])
      legal_actions.extend([43, 47, 51, 60])
      legal_actions.extend([63, 67, 71, 80])
    if res[4] > 0:
      legal_actions.extend([4, 8, 12, 16])
      legal_actions.extend([24, 28, 32, 36])
      legal_actions.extend([44, 48, 52, 56])
      legal_actions.extend([64, 68, 72, 76])
    
    return legal_actions

In [20]:
class Ruler:
  def __init__(self, id):
    self.id = id
    self.harvest = np.ones(5) #Format: 'w', 'h', 'o', 't', 'b'
    self.upgrade = np.ones(5) #Format: 'w', 'h', 'o', 't', 'b'
    self.loyalty = np.ones(5)*50 #Loyalty relations with other rulers
    self.last_traded = None
    self.level = 1
    self.resources = np.zeros(5)
    self.rounds_played = 0
    self.last_action = None
    self.last_reward = None
    self.succ_trades = 0
    self.trade_attempts = 0
    self.avg_reward = 0

    self.epsilon = 1.0 #Exploration factor
    self.optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    self.memory = ReplayBuffer() #Stores memory of one whole game
    self.gamma = 0.9 #Discount rate
    self.model = build_q_network(9, 81)
    self.model1 = keras.models.clone_model(self.model)
    self.model1.build(self.model.input_shape)
    self.model1.set_weights(self.model.get_weights())

  def init_beatles(self):
    #Requirements are temporarily modified to make initial learning easier
    if self.id == 0: #John
      self.harvest = np.array([0, 0, 1, 1, 0])
      self.upgrade = np.array([0, 1, 0, 1, 1])
    elif self.id == 1: #Ringo
      self.harvest = np.array([1, 0, 0, 0, 1])
      self.upgrade = np.array([1, 0, 1, 0, 1])
    elif self.id == 2: #George
      self.harvest = np.array([1, 1, 0, 0, 0])
      self.upgrade = np.array([1, 1, 1, 0, 0])
    elif self.id == 3: #Paul
      self.harvest = np.array([0, 1, 0, 0, 1])
      self.upgrade = np.array([0, 0, 1, 1, 1])
    elif self.id == 4: #Pete
      self.harvest = np.array([1, 0, 1, 0, 0])
      self.upgrade = np.array([0, 1, 0, 1, 1])



  def consider_proposal(self, proposal): #Harcoded rule for accepting or rejecting a proposed trade

    #Proposal --> [id of proposing agent, resource being demanded, resource being given]
    if self.resources[proposal[1]] < 1:
      return False

    if self.upgrade[proposal[2]] > 0: # and self.resources[proposal[2]] < self.resources[proposal[2]]:
      if self.harvest[proposal[1]] > 0:
        if self.harvest[proposal[2]] == 0:
          return True #Needs the given resource, does not harvest it, and harvests the proposed resource
        else:
          if np.random.random()*100 < self.loyalty[proposal[0]]:
            return True
          else:
            return False #Random selection
      else:
        if self.loyalty[proposal[0]] > 90 and self.upgrade[proposal[1]] == 0:
          return True #Feels obliged due to loyalty relations, and does not need the demanded resource
        else:
          return False

    else:
      return False

  def harvest_resources(self):
    self.resources = self.resources + self.harvest

  def upgrade_level(self):
    check = np.all(self.resources - self.upgrade >= 0)
    if check:
      self.resources = self.resources - self.upgrade
      self.level = self.level + 1

    return check


  def take_action(self, all_rulers):
    reward = 0
    state = np.zeros(9)
    state[0:5] = self.resources
    i = 5
    for j in range(5):
      if j == self.id:
        continue
      state[i] = self.loyalty[j]
      i = i + 1

    legal_actions = get_legal_actions(self.resources)
    actions_np = (self.model(state.reshape(1, 9))).numpy()
    for action in range(1, 81):
      if action not in legal_actions:
        actions_np[0, action] = -np.inf #Illegal actions should get -inf Q-value
    actions = tf.convert_to_tensor(actions_np)

    if np.random.random() < self.epsilon:
      action = np.random.choice(legal_actions)
    else:
      action = np.argmax(actions)
    self.epsilon = self.epsilon*0.99
    self.epsilon = np.clip(self.epsilon, 0.1, 1.0)

    self.last_action = action
      
    if action == 0: #Pure harvesting
      self.harvest_resources()
      for p in range(5):
          if self.upgrade[p] > 0 and self.harvest[p] > 0:
              reward = reward + 5
    else: #Trading
      self.trade_attempts = self.trade_attempts + 1
      mult = (action-1)//20
      traded_resource = traded_resources((action-1)%20)
      proposal = [self.id, traded_resource[0], traded_resource[1]]
      og = self.resources[traded_resource[0]]
      flag = all_rulers[(self.id+mult+1)%5].consider_proposal(proposal)
      check = conduct_trade(self, all_rulers[(self.id+mult+1)%5], proposal, flag)
      new = self.resources[traded_resource[0]]
      if self.upgrade[traded_resource[0]] > 0:
        reward = reward + (new-og)*7
      if flag:
        reward = reward + 2
        self.succ_trades = self.succ_trades + 1
      else:
        reward = reward - 0.1

    if self.upgrade_level():
      reward = reward + 20
    else:
      reward = reward + 0

    self.last_reward = reward
    self.avg_reward = (self.avg_reward*(self.rounds_played-1) + reward)/(self.rounds_played)
    
    next_state = np.zeros(9)
    next_state[0:5] = self.resources
    i = 5
    for j in range(5):
      if j == self.id:
        continue
      next_state[i] = self.loyalty[j]
      i = i + 1

    next_legal_actions = get_legal_actions(self.resources)
      
    return (state, action, reward, next_state, next_legal_actions)

  @tf.function  
  def train(self):

    # Sample replay buffer
    states, actions, rewards, next_states, many_legal_actions, done_check = self.memory.sample(20)
    states = tf.convert_to_tensor(states, dtype=tf.float32)
    next_states = tf.convert_to_tensor(next_states, dtype=tf.float32)
    actions = tf.convert_to_tensor(actions, dtype=tf.int32)
    rewards = tf.convert_to_tensor(rewards, dtype=tf.float32)
    done_check = tf.convert_to_tensor(done_check, dtype=tf.float32)
      
    legal_mask = np.zeros((20, 81), dtype=bool)

    for i, legal_actions in enumerate(many_legal_actions):
        legal_mask[i, legal_actions] = True

    legal_mask = tf.convert_to_tensor(legal_mask)
        
    
    # Update target network periodically
    if self.rounds_played % 10 == 0:
        self.model1.set_weights(self.model.get_weights())

    with tf.GradientTape() as tape:

        # -----------------------------
        # Online network
        # -----------------------------
        q_values = self.model(states, training=True)

        # Select only the Q-values corresponding to the chosen actions
        chosen_q = tf.gather(
            q_values,
            actions,
            axis=1,
            batch_dims=1
        )

        # -----------------------------
        # Double DQN target
        # -----------------------------
        next_online_q = self.model(next_states, training=False)
     
        masked_online_q = tf.where(
            legal_mask,
            next_online_q,
            tf.fill(tf.shape(next_online_q), -1e9)
        )
        
        best_actions = tf.argmax(masked_online_q, axis=1)
        
        next_target_q = self.model1(next_states, training=False)

        target_q = tf.gather(
            next_target_q,
            best_actions,
            axis=1,
            batch_dims=1
        )

        # Don't bootstrap terminal states
        done = tf.cast(done_check, tf.float32)

        targets = rewards + (1.0 - done) * self.gamma * target_q

        # Stop gradients through the target network
        targets = tf.stop_gradient(targets)

        # Mean squared TD error
        loss = tf.reduce_mean(tf.square(chosen_q - targets))

    grads = tape.gradient(loss, self.model.trainable_variables)
    self.optimizer.apply_gradients(
        zip(grads, self.model.trainable_variables)
    )

    avg_q = tf.reduce_mean(chosen_q)

    return (
        loss,
        avg_q,
        actions
    )

In [6]:
def run_round(all_rulers):
  for i in range(5):
    done = False
    if all_rulers[i].rounds_played == 499:
      done = True
    all_rulers[i].rounds_played = all_rulers[i].rounds_played + 1 #Increments round counter
    state, action, reward, next_state, legal_actions = all_rulers[i].take_action(all_rulers)
    all_rulers[i].memory.store(state, action, reward, next_state, legal_actions, done) #Stores in memory buffer

def jealousy_check(all_rulers):
  bf = []
  for ruler in all_rulers:
    bf.append(np.argmax(ruler.loyalty))
  for i in range(len(bf)):
    if bf[bf[i]] != i:
      all_rulers[i].loyalty[bf[i]] = all_rulers[i].loyalty[bf[i]]*0.7

def train_all(all_rulers):
  losses = []
  q_vals = []
  all_actions = []
  for ruler in all_rulers:
    loss = 0
    for _ in range(5):
      train_loss, q_val_mean, actions = ruler.train()
      loss = loss + float(train_loss)
      q_vals.append(float(q_val_mean))
    loss = loss/5
    losses.append(loss)
    all_actions.append(actions)
  return losses, q_vals, all_actions #actions is of the last of the 5 runs


def bound_resources(all_rulers):
  for ruler in all_rulers:
    if np.sum(ruler.resources) >= 10:
      ruler.resources = -(ruler.resources//-2)

Now we run a whole game, consisting of 500 rounds.

In [7]:
def run_game(all_rulers):
  for ruler in all_rulers:
    ruler.resources = np.zeros(5)
    ruler.level = 1
    ruler.rounds_played = 0
    ruler.epsilon = 1.0
    ruler.succ_trades = 0
    ruler.trade_attempts = 0
    ruler.avg_reward = 0
    ruler.loyalty = np.ones(5)*50
  for i in range(200):
    bound_resources(all_rulers)
    run_round(all_rulers)
    #jealousy_check(all_rulers)
    if i > 19:
      losses, q_vals, all_actions = train_all(all_rulers)
      print(i)
      print(losses)
      print(all_rulers[0].avg_reward)
      print(all_rulers[1].avg_reward)
      print(all_rulers[2].avg_reward)
      print(all_rulers[3].avg_reward)
      print(all_rulers[4].avg_reward)
      print('\n')
      #print(q_vals)

Let's create certain characters, who get specific resources, but need more than just those to upgrade their kingdoms. This forces them to trade with the others to succeed. The Beatles

In [22]:
all_rulers = [Ruler(0), Ruler(1), Ruler(2), Ruler(3), Ruler(4)]
for player in all_rulers:
  player.init_beatles()

In [23]:
all_rulers[0].model.load_weights('ruler1_weights.weights.h5')
all_rulers[1].model.load_weights('ruler2_weights.weights.h5')
all_rulers[2].model.load_weights('ruler3_weights.weights.h5')
all_rulers[3].model.load_weights('ruler4_weights.weights.h5')
all_rulers[4].model.load_weights('ruler5_weights.weights.h5')

In [27]:
run_game(all_rulers)

20
[20909.629296875, 6541.83525390625, 5688.8572265625, 5006.80625, 1342.908935546875]
0.9523809523809523
0.9523809523809523
0.5238095238095238
0.5238095238095238
0.5238095238095238


21
[19784.7546875, 6194.8318359375, 5240.73837890625, 4538.7302734375, 1674.016552734375]
0.9090909090909091
0.9090909090909091
0.5
0.5909090909090909
0.5


22
[18662.30625, 5731.78896484375, 4820.23330078125, 4106.30302734375, 1646.620849609375]
0.8695652173913043
0.8695652173913043
0.4782608695652174
0.5652173913043478
0.4782608695652174


23
[17594.653515625, 5216.65234375, 4427.736328125, 3707.439697265625, 1509.10703125]
0.8333333333333334
0.8333333333333334
0.4583333333333333
0.5416666666666666
0.4583333333333333


24
[16582.3150390625, 4737.8015625, 4063.03994140625, 3342.762060546875, 1381.3731201171875]
0.8
0.8
0.44
0.52
0.52


25
[15624.8744140625, 4330.092578125, 3226.259912109375, 3010.127197265625, 1263.4531494140624]
1.5384615384615385
0.7692307692307693
1.5384615384615385
0.5769230769230769

In [44]:
def print_levels(all_rulers):
  print("John's level: " + str(all_rulers[0].level))
  print("Ringo's level: " + str(all_rulers[1].level))
  print("George's level: " + str(all_rulers[2].level))
  print("Paul's level: " + str(all_rulers[3].level))
  print("Pete's level: " + str(all_rulers[4].level))

print_levels(all_rulers)

John's level: 1
Ringo's level: 3
George's level: 1
Paul's level: 1
Pete's level: 1


In [45]:
all_rulers[0].model.save_weights('ruler1_weights.weights.h5')
all_rulers[1].model.save_weights('ruler2_weights.weights.h5')
all_rulers[2].model.save_weights('ruler3_weights.weights.h5')
all_rulers[3].model.save_weights('ruler4_weights.weights.h5')
all_rulers[4].model.save_weights('ruler5_weights.weights.h5')

In [46]:
def run_test_game(all_rulers):
    all_rulers[0].model.load_weights('ruler1_weights.weights.h5')
    all_rulers[1].model.load_weights('ruler2_weights.weights.h5')
    all_rulers[2].model.load_weights('ruler3_weights.weights.h5')
    all_rulers[3].model.load_weights('ruler4_weights.weights.h5')
    all_rulers[4].model.load_weights('ruler5_weights.weights.h5')
    for ruler in all_rulers:
        ruler.epsilon = 0
        ruler.resources = np.zeros(5)
        ruler.level = 1
        ruler.rounds_played = 0
        ruler.succ_trades = 0
        ruler.trade_attempts = 0
        ruler.avg_reward = 0
        
    for i in range(200):
        bound_resources(all_rulers)
        run_round(all_rulers)
        for ruler in all_rulers:
            print(ruler.last_action)
            print(ruler.last_reward)
            print('\n')

run_test_game(all_rulers)
for ruler in all_rulers:
    ruler.epsilon = 1.0

0
5


0
10


0
10


0
5


0
0


75
-0.1


36
-0.1


29
-0.1


14
-0.1


45
9.0


27
-0.1


36
-0.1


29
-0.1


52
-0.1


38
-0.1


15
-0.1


36
-0.1


29
-0.1


14
-0.1


38
-0.1


75
-0.1


36
-0.1


29
-0.1


14
-0.1


18
-0.1


75
-0.1


36
-0.1


29
-0.1


14
-0.1


38
-0.1


71
2


36
-0.1


17
-0.1


14
-0.1


67
9.0


75
-0.1


36
-0.1


65
-0.1


28
-0.1


38
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


38
-0.1


75
-0.1


77
-0.1


29
-0.1


28
-0.1


38
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


54
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


38
-0.1


75
-0.1


72
9.0


29
-0.1


28
-0.1


38
-0.1


75
-0.1


59
-0.1


29
-0.1


28
-0.1


38
-0.1


75
-0.1


0
30


29
-0.1


28
-0.1


38
-0.1


75
-0.1


0
10


29
-0.1


28
-0.1


38
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


70
-0.1


75
-0.1


44
-0.1


29
-0.1


28
-0.1


34
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


70
-0.1


75
-0.1


36
-0.1


29
-0.1


28
-0.1


38
-0.1


75
-0.1


36
-0.1


29
-0.1




In [47]:
def get_metrics(all_rulers):
    for ruler in all_rulers:
        print([ruler.id, ruler.avg_reward, (ruler.succ_trades/ruler.trade_attempts)])

get_metrics(all_rulers)

[0, np.float64(0.2044999999999992), 0.020512820512820513]
[1, np.float64(0.9415000000000043), 0.015789473684210527]
[2, np.float64(0.25299999999999967), 0.015228426395939087]
[3, np.float64(-0.06399999999999985), 0.005025125628140704]
[4, np.float64(0.15899999999999945), 0.025380710659898477]


In [ ]:
for l in range(1000):
    run_game(all_rulers)
    get_metrics(all_rulers)
    print(l)

20
[0.00018440044514136388, 2.615459688968258e-05, 1.2707539781928063, 12.263086795806885, 1.1730786695807182e-06]
1.9714285714285713
0.4809523809523813
2.8285714285714274
1.0190476190476185
0.11428571428571424


21
[0.00013437048182822763, 1.0901558505338471e-05, 0.11560496091842651, 13.010082912445068, 4.145920826204019e-07]
1.8772727272727272
0.8681818181818186
2.6954545454545444
0.9681818181818176
1.0136363636363634


22
[6.583780523214955e-05, 4.520127572504862e-06, 0.4321769416332245, 4.864002704620361, 4.215522210415656e-07]
1.7913043478260868
0.8260869565217395
2.5739130434782598
1.1434782608695646
1.3608695652173912


23
[4.4704596257361116e-05, 5.9990356930939015e-06, 0.6094572082161903, 12.272257328033447, 1.9334920366986808e-07]
1.7124999999999997
0.7875000000000002
2.462499999999999
1.0916666666666661
1.2999999999999998


24
[3.1410465817316434e-05, 1.894290375048513e-06, 0.4340862721204758, 13.02361536026001, 8.969956439841553e-08]
1.6399999999999997
0.7520000000000002
2.